# Land Cover Chips Extraction
## GEOG 6160 Final Project
## Magnus Tveit

**Run `PrepData.ipynb` first** to ensure `Data/` is populated.

Reads polygons from the shapefile and extracts 8×8 pixel chips from each
training year's 7-band Landsat stack. Chips are saved as `.npy` files in the folder
structure expected by `LandCoverCNN.ipynb`.

The shapefile was created in ArcGIS pro

**Run `PrepData.ipynb` first** to ensure `Data/` is populated.

Class schema:
```
type : 0 = Lake           (Lake Powell)
       1 = River          (above delta)
       2 = Sediment Bank  (exposed Lake Powell sediment)
       3 = Rock           (rock or soil in the surrounding areas)
```

Shapefile field schema:
```
type : 0–3 as above
year : 13 / 16 / 20 / 22 = year-specific polygon 
```

Output structure:
```
Chips/
   train/
      Lake/
      River/
      Sediment Bank/
      Rock/
   valid/
      ...
   test/
      ...
```

> **Note:** Delete `Chips/` before re-running if you change attempt number or chip size.

### Set Up

In [ ]:
import os, random
import numpy as np
import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd

### Settings
> Note: Update the `attempt` to match the `TrainingPolygons/` subfolder

In [ ]:
# Attempt number; relates to the folder in which the training polygons Shapefiles are stored
attempt = 3

random.seed(42)
np.random.seed(42)

base_dir  = os.getcwd()
data_dir  = os.path.join(base_dir, "..", "Data")
shp_path  = os.path.join(base_dir, "TrainingPolygons", str(attempt), "TrainingPolygons.shp")
chips_dir = os.path.join(base_dir, "Chips")

CHIP_SIZE               = 8
COVERAGE_THRESHOLD      = 0.3   # chip must be ____ inside the polygon to be kept
MAX_MINORITY_MULTIPLIER = 4     # augment minority classes to 4x the median class count
TRAINING_YEARS          = [13, 16, 20, 22]
CLASSES                 = {0:"Lake", 1:"River", 2:"Sediment Bank", 3:"Rock"}
BAND_SUFFIXES           = ["_SR_B2.TIF", "_SR_B3.TIF", "_SR_B4.TIF",
                            "_SR_B5.TIF", "_SR_B6.TIF", "_SR_NDVI.TIF", "_SR_NDWI.TIF"]
SCALE, OFFSET           = 0.0000275, -0.2   # Landsat Collection 2 surface reflectance scaling
TRAIN_RATIO, VALID_RATIO = 0.70, 0.15

# Create output folder structure
for split in ["train", "valid", "test"]:
    for cls in CLASSES.values():
        os.makedirs(os.path.join(chips_dir, split, cls), exist_ok=True)

os.environ["SHAPE_RESTORE_SHX"] = "YES"   # auto-rebuild missing .shx index
gdf = gpd.read_file(shp_path).to_crs("EPSG:32612")
print(f"  {len(gdf)} polygons loaded")
print(f"  year values : {sorted(gdf['year'].unique())}")
print(f"  type values : {sorted(gdf['type'].unique())}\n")

In [ ]:
import json
config = {
    "CHIP_SIZE"          : CHIP_SIZE,
    "COVERAGE_THRESHOLD" : COVERAGE_THRESHOLD,
}
with open(os.path.join(base_dir, "experiment_config.json"), "w") as f:
    json.dump(config, f, indent=2)
print("experiment_config.json written.")

### Extract Chips
Slides an 8×8 grid across each polygon and keeps chips with >50% coverage. Delta chips are augmented ×6 to try to help balance the dataset.

In [ ]:
def extract_chips(stacked, transform, polygon, chip_size, coverage_threshold):
    """Yield (chip_size, chip_size, 7) chips whose centre falls inside the polygon."""
    n_bands, H, W = stacked.shape
    half = chip_size // 2

    # Rasterize the polygon to a boolean mask
    inside = geometry_mask([polygon.__geo_interface__],
                           out_shape=(H,W), transform=transform, invert=True)
    rows, cols = np.where(inside)
    if len(rows) == 0: return

    seen = set()
    for r, c in zip(rows, cols):
        # Snap to the nearest grid-aligned chip origin
        r_s = (r // half) * half
        c_s = (c // half) * half
        if (r_s, c_s) in seen: continue
        seen.add((r_s, c_s))
        r0,r1 = r_s-half, r_s+half
        c0,c1 = c_s-half, c_s+half
        if r0<0 or c0<0 or r1>H or c1>W: continue
        if inside[r0:r1, c0:c1].mean() < coverage_threshold: continue
        yield np.moveaxis(stacked[:, r0:r1, c0:c1], 0, -1)


def augment_chip(chip):
    """Return 6 versions of a chip: original + 3 rotations + 2 flips."""
    aug = [chip]
    for k in [1,2,3]: aug.append(np.rot90(chip, k=k, axes=(0,1)))
    aug.append(np.flip(chip, axis=0))
    aug.append(np.flip(chip, axis=1))
    return aug


year_folders = sorted([f for f in os.listdir(data_dir)
                        if os.path.isdir(os.path.join(data_dir, f))])
print(f"Found {len(year_folders)} year folders: {year_folders}")
print(f"Training years: {TRAINING_YEARS}\n")

all_chips = {cls: [] for cls in CLASSES.values()}

for year_folder in year_folders:
    year_code = int(year_folder[-2:])
    if year_code not in TRAINING_YEARS:
        print(f"{year_folder}: prediction-only — skipping"); continue

    year_dir  = os.path.join(data_dir, year_folder)
    tif_files = os.listdir(year_dir)

    # Select polygons for this year (year-specific + permanent year=00 polygons)
    year_gdf = gdf[gdf["year"].isin([year_code, 0])]
    if year_gdf.empty:
        print(f"{year_folder}: no polygons — skipping"); continue

    # Locate all 7 band files
    band_paths = []
    for suffix in BAND_SUFFIXES:
        match = next((f for f in tif_files if f.endswith(suffix)), None)
        if match is None:
            print(f"{year_folder}: missing {suffix} — skipping"); break
        band_paths.append(os.path.join(year_dir, match))
    if len(band_paths) != 7: continue

    with rasterio.open(band_paths[0]) as src:
        transform = src.transform

    # Stack and scale all 7 bands
    bands = []
    for i, bp in enumerate(band_paths):
        with rasterio.open(bp) as src:
            b = src.read(1).astype("float32")
        b = np.clip(b*SCALE+OFFSET,0,1) if i<5 else np.clip(b,-1,1)
        bands.append(b)
    stacked = np.stack(bands, axis=0)

    year_counts = {cls: 0 for cls in CLASSES.values()}
    for _, row in year_gdf.iterrows():
        cls_name = CLASSES[int(row["type"])]
        for chip in extract_chips(stacked, transform, row.geometry,
                                  CHIP_SIZE, COVERAGE_THRESHOLD):
            all_chips[cls_name].append(chip)
            year_counts[cls_name] += 1

    print(f"{year_folder}: " + "  ".join(f"{k}={v}" for k,v in year_counts.items()))

print("\nChip counts before balancing:")
for cls, chips in all_chips.items():
    print(f"  {cls:12s}: {len(chips)}")

### Balancing
Augment minority classes up to MAX_MINORITY_MULTIPLIER × median count

In [ ]:
counts   = {cls: len(chips) for cls, chips in all_chips.items() if len(chips) > 0}
median_n = sorted(counts.values())[len(counts) // 2]
print(f"Median class count (subsample cap): {median_n}\n")

balanced = {}
for cls, chips in all_chips.items():
    n = len(chips)
    if n == 0:
        balanced[cls] = []
    elif n > median_n:
        random.shuffle(chips)
        balanced[cls] = chips[:median_n]   # oversized: subsample down
    else:
        balanced[cls] = list(chips)        # undersized: keep all, CNN will augment

print("Chip counts after balancing:")
for cls, chips in balanced.items():
    print(f"  {cls:12s}: {len(chips)}")

### Save
Shuffle and split each class into train / valid / test, then write to disk.

In [ ]:
import shutil
for split in ["train", "valid", "test"]:
    for cls in CLASSES.values():
        folder = os.path.join(chips_dir, split, cls)
        if os.path.exists(folder):
            shutil.rmtree(folder)
        os.makedirs(folder)
print("Chips/ cleared and rebuilt.\n")

chip_counter = 0
for cls_name, chips in balanced.items():
    random.shuffle(chips)
    n       = len(chips)
    n_train = int(n * TRAIN_RATIO)
    n_valid = int(n * VALID_RATIO)
    splits  = {
        "train": chips[:n_train],
        "valid": chips[n_train:n_train+n_valid],
        "test" : chips[n_train+n_valid:]
    }
    for split, split_chips in splits.items():
        out_dir = os.path.join(chips_dir, split, cls_name)
        for chip in split_chips:
            np.save(os.path.join(out_dir, f"{cls_name}_{chip_counter:06d}.npy"), chip)
            chip_counter += 1
        print(f"  {split:5s} / {cls_name:10s} : {len(split_chips)} chips")

print("\nDone. Ready for LandCoverCNN.ipynb")